# PerfectPetClub - Data Engineering ETL Pipeline
This notebook  Extract, Transform and Loads supplied database.

### The pipeline:
Extracts records from a CSV file.

Cleans and standardises extracted data.

Exports the cleaned dataset in CSV and Parquet formats.

Displays the original and transformed datasets for validation.


In [47]:
from pathlib import Path
import sys
import pandas as pd

sys.path.insert(0, str(Path("../src").resolve()))

%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [52]:

from crm import run_crm_etl

# Call the function to run  crm etl file 
crm_df, cleaned_crm_df,crm_features_df = run_crm_etl()

# Display the first five rows to confirm to check extraction and transformation 
display(crm_df.head())

display(cleaned_crm_df.head())

display(crm_features_df.head())

Missing or invalid dates: 0
CRM ETL and feature engineering completed successfully!


,customer_id,recieved_datatime,platform,type
0,55,2026-01-01 00:42:47.000,['e-mail'],complaint
1,49,2026-01-01 01:25:34.000,['live-chat'],query
2,44,2026-01-01 03:33:55.001,['telephone'],query
3,33,2026-01-01 05:42:16.002,['e-mail'],complaint
4,33,2026-01-01 06:25:03.002,['e-mail'],complaint


,customer_id,received_datetime,platform,type
0,55,2026-01-01 00:42:47.000,e-mail,complaint
1,49,2026-01-01 01:25:34.000,live-chat,query
2,44,2026-01-01 03:33:55.001,telephone,query
3,33,2026-01-01 05:42:16.002,e-mail,complaint
4,33,2026-01-01 06:25:03.002,e-mail,complaint


,customer_id,total_interactions,complaint_count
0,1,3,2
1,2,45,21
2,3,167,90
3,4,281,132
4,5,5,1


In [49]:

 
# Test: Can our validation detect an invalid date?

test_data = raw_customers.copy()

# Deliberately insert an invalid subscription date
test_data.loc[0, 'sub_end_date'] = 'invalid-date'

try:
    validate_customer_data(test_data)
    print("TEST FAILED: Invalid date was accepted.")

except AssertionError as e:
    print("TEST PASSED: Invalid date was detected.")
    print(e)



 

NameError: name 'validate_customer_data' is not defined

In [ ]:
# 1. Transform customer features
clean_customers = transform_customer_features(raw_customers)

# 2. Merge all features on customer_id (1-to-1 relationship)
master_features = clean_customers.merge(raw_sales_reviews, on='customer_id', how='left')

# 3. Clean up PII for GDPR compliance (drop name, full postcode, and raw subscription IDs)
final_dataset = master_features.drop(columns=['customer_name', 'postcode', 'subscription_id'])

print(f'Master feature dataset shape: {final_dataset.shape}')
final_dataset.head()


In [ ]:

# Churn rate by subscription tier
tier_summary = final_dataset.groupby('subscription_tier')['is_churned'].agg(
    total_customers='count',
    churn_rate='mean'
).reset_index()

tier_summary['churn_rate'] = (tier_summary['churn_rate'] * 100).round(1).astype(str) + '%'
print('--- Churn Rate by Subscription Tier ---')
print(tier_summary)

# Churned vs Retained spend & review comparison
print('\n--- Mean Metrics by Churn Status ---')
print(final_dataset.groupby('is_churned')[['total_spend_gbp', 'total_orders', 'avg_star_rating', 'one_star_reviews_count']].mean().round(2))
Stage 4: Load — Export Curated Dataset to Apache Parquet
As specified in Task 2, the final transformed dataset must be exported as an Apache Parquet file. Parquet provides columnar compression, preserves column data types, and is optimized for downstream analytical queries.



In [ ]:
# Create output directory
output_dir = Path('../output')
output_dir.mkdir(parents=True, exist_ok=True)

parquet_path = output_dir / 'perfectpetclub_churn_features.parquet'

# Export to Parquet
final_dataset.to_parquet(parquet_path, index=False, engine='pyarrow')

print(f'✓ Successfully loaded {len(final_dataset)} records into Parquet format:')
print(f'  Location: {parquet_path.resolve()}')
print(f'  File size: {parquet_path.stat().st_size:,} bytes')


In [ ]:
# Verification: Read back parquet file to guarantee integrity
df_verified = pd.read_parquet(parquet_path)
print(f'Parquet verification passed: {df_verified.shape[0]} rows, {df_verified.shape[1]} columns.')
print('\nDataset Columns:')
for col in df_verified.columns:
    print(f' - {col} ({df_verified[col].dtype})')

In [ ]:
# Final verification of feature engineering and Parquet output

# Check 1: Customer records have not been duplicated
assert final_dataset['customer_id'].is_unique, "Duplicate customers found"

# Check 2: Customer count remains unchanged
assert len(final_dataset) == len(raw_customers), "Customer count mismatch"

# Check 3: Churn calculation is correct
expected_churn = (final_dataset['auto_renew'] == 0).astype(int)
assert (final_dataset['is_churned'] == expected_churn).all(), "Incorrect churn values"

# Check 4: Subscription durations are not negative
assert (final_dataset['subscription_tenure_days'] >= 0).all(), "Negative duration found"

# Check 5: Parquet file contains the expected records and columns
assert df_verified.shape == final_dataset.shape, "Parquet dataset shape mismatch"
assert list(df_verified.columns) == list(final_dataset.columns), "Parquet columns mismatch"

print("PASS: Customer records are unique")
print("PASS: Customer count is correct")
print("PASS: Churn calculations are correct")
print("PASS: Subscription durations are valid")
print("PASS: Parquet structure is verified")